# Run on Google Colab

Single notebook for the entire pipeline. Toggle the `RUN_STAGE_*` flags in
the configuration cell to choose which stages to run.

Stages:
- A   - SimCLR self-supervised pretraining (produces `encoder_pretrained.pt`)
- B.1 - Linear probe with the frozen encoder (produces `linear_probe_best.pt`)
- B.2 - Full fine-tuning (produces `finetune_best.pt`)
- C   - CIFAR-10-C evaluation of the fine-tuned model (15 corruptions x 5 severities)

Stage dependencies (when a stage is skipped, the upstream artifact must already
exist on disk - the notebook restores from Google Drive when available):
- B.1 requires Stage A's `encoder_pretrained.pt`
- B.2 requires Stage A's `encoder_pretrained.pt`
- C   requires Stage B.2's `finetune_best.pt`


## 1. Configuration

Edit this cell only. Everything below it is generic.

Pick a `CONFIG_PRESET`:
- `"default"` — overnight-grade run (200 ep SimCLR, 30 ep finetune, full TTT eval).
- `"smoke"`   — 2 epochs each stage, ~5 min sanity check before any real run.

Tip: switch Colab to GPU first (`Runtime -> Change runtime type -> L4 / A100`).


In [ ]:
# --- Repository ---
REPO_URL = "https://github.com/3v3r51nc3/SelfSupervised-TTT-ImageClassification.git"
BRANCH = "main"
PROJECT_DIR = "/content/SelfSupervised-TTT-ImageClassification"

# --- Config selection ---
# Pick a preset to load. Add new entries to CONFIG_PATHS as you create configs.
#   "default" -> configs/default.yaml  (overnight run, ~5h on L4)
#   "smoke"   -> configs/smoke.yaml    (2 epochs each, ~5 min sanity check)
CONFIG_PRESET = "default"
CONFIG_PATHS = {
    "default": "configs/default.yaml",
    "smoke":   "configs/smoke.yaml",
}
CONFIG_PATH = CONFIG_PATHS[CONFIG_PRESET]

# --- Persistence (Google Drive) ---
USE_GOOGLE_DRIVE = True
DRIVE_RUNS_DIR = "/content/drive/MyDrive/selfsupervised_ttt_runs"

# Restore prior logs/checkpoints from Drive before training.
# Required when running a stage whose upstream artifact is not present locally.
RESTORE_FROM_DRIVE = True

# --- Stage selection ---
RUN_STAGE_A = True
RUN_STAGE_B1 = True
RUN_STAGE_B2 = True
RUN_STAGE_C = True

# --- Config overrides (None = keep YAML default) ---
# Any value set here replaces the corresponding field in the loaded YAML
# without editing the file. Use "section.field" keys.
CONFIG_OVERRIDES = {
    # Data
    "data.num_workers": None,
    "data.batch_size_ssl": None,
    "data.batch_size_sup": None,
    "data.augment_supervised": None,
    # SimCLR (Stage A)
    "simclr.epochs": None,
    "simclr.learning_rate": None,
    "simclr.temperature": None,
    "simclr.warmup_epochs": None,
    "simclr.use_amp": None,
    # Linear probe (Stage B.1)
    "linear_probe.epochs": None,
    "linear_probe.learning_rate": None,
    # Fine-tune (Stage B.2)
    "finetune.epochs": None,
    "finetune.learning_rate": None,
    "finetune.weight_decay": None,
    "finetune.label_smoothing": None,
    "finetune.early_stopping_patience": None,
    # TTT (Stage C)
    "ttt.enabled": None,
    "ttt.steps": None,
    "ttt.learning_rate": None,
    "ttt.adapt_scope": None,
    # Experiment / misc
    "experiment.name": None,
    "experiment.seed": None,
}


## 2. Clone Repository and Install Dependencies


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if Path(PROJECT_DIR).exists():
    print(f"Removing existing directory: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")


In [ ]:
!nvidia-smi || true
!python --version
# Colab pre-installs torch/torchvision/numpy/PyYAML/tensorboard/pandas.
# Only timm is missing.
!pip install -q timm


## 3. Restore Previous Run from Google Drive (Optional)

Copies `logs/` and `checkpoints/` back from Drive so that downstream stages
can find the artifacts they need (encoder, fine-tune checkpoint).


In [ ]:
if USE_GOOGLE_DRIVE and RESTORE_FROM_DRIVE:
    drive_runs = Path(DRIVE_RUNS_DIR)
    for sub in ("logs", "checkpoints"):
        src = drive_runs / sub
        if src.exists():
            shutil.copytree(src, sub, dirs_exist_ok=True)
            print(f"Restored {src} -> {sub}")
        else:
            print(f"Nothing to restore from {src}")
else:
    print("Skipping restore (RESTORE_FROM_DRIVE is False or Drive disabled).")


## 4. Get CIFAR-10-C (only if Stage C will run)

On the first run: install `aria2`, download the ~2.9 GB tar from Zenodo with
8 parallel connections, then cache it to Google Drive at
`<DRIVE_RUNS_DIR>/cache/CIFAR-10-C.tar`.

On every later run: restore the tar from Drive (~30 s) instead of
re-downloading. The local extracted directory makes the cell a no-op once
data is already on disk.


In [ ]:
from IPython import get_ipython
ipy = get_ipython()

if RUN_STAGE_C:
    cifar10c_dir = Path("data/CIFAR-10-C")
    sentinel = cifar10c_dir / "labels.npy"
    archive = Path("data/CIFAR-10-C.tar")
    drive_archive = Path(DRIVE_RUNS_DIR) / "cache" / "CIFAR-10-C.tar"

    if sentinel.exists():
        print(f"CIFAR-10-C already extracted at {cifar10c_dir}")
    else:
        cifar10c_dir.parent.mkdir(parents=True, exist_ok=True)

        if not archive.exists():
            # 1) Restore the tar from Drive cache when available.
            if USE_GOOGLE_DRIVE and drive_archive.exists():
                print(f"Restoring CIFAR-10-C.tar from Drive: {drive_archive}")
                shutil.copy2(drive_archive, archive)
            else:
                # 2) Otherwise install aria2 and download in parallel from Zenodo.
                print("Installing aria2 ...")
                rc = ipy.system("apt-get install -y -q aria2")
                if rc not in (None, 0):
                    raise RuntimeError(f"apt-get install aria2 failed (rc={rc})")
                url = "https://zenodo.org/records/2535967/files/CIFAR-10-C.tar"
                print("Downloading CIFAR-10-C from Zenodo with 8 parallel connections ...")
                rc = ipy.system(
                    "aria2c -x 8 -s 8 --console-log-level=warn "
                    "--summary-interval=5 -o CIFAR-10-C.tar -d data "
                    f"{url}"
                )
                if rc not in (None, 0):
                    raise RuntimeError(f"aria2c failed (rc={rc})")

                # 3) Cache the freshly downloaded tar on Drive for future sessions.
                if USE_GOOGLE_DRIVE:
                    drive_archive.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(archive, drive_archive)
                    print(f"Cached archive to {drive_archive}")

        rc = ipy.system(f"tar -xf {archive} -C data")
        if rc not in (None, 0):
            raise RuntimeError(f"tar failed with exit code {rc}")
        print(f"Extracted CIFAR-10-C to {cifar10c_dir}")
else:
    print("Stage C is disabled; skipping CIFAR-10-C download.")


## 5. Inspect the Config

All stages read from the same YAML. For a smoke test, set the relevant
`epochs` fields to 1.


In [ ]:
!sed -n "1,220p" {CONFIG_PATH}


## 6. Run the Selected Stages

Each stage is gated by its `RUN_STAGE_*` flag. When a stage is skipped, the
notebook checks that the artifact it would have produced is already on disk.


In [ ]:
from dataclasses import replace

from src.core.config import ConfigLoader
from src.core.pipeline import ExperimentPipeline

config = ConfigLoader.load(CONFIG_PATH)

# Apply notebook overrides on top of the YAML defaults.
_grouped: dict[str, dict] = {}
for _key, _value in CONFIG_OVERRIDES.items():
    if _value is None:
        continue
    _section, _field = _key.split(".", 1)
    _grouped.setdefault(_section, {})[_field] = _value
if _grouped:
    _new_sections = {s: replace(getattr(config, s), **f) for s, f in _grouped.items()}
    config = replace(config, **_new_sections)
    print("Applied overrides:")
    for _s, _f in _grouped.items():
        for _k, _v in _f.items():
            print(f"  {_s}.{_k} = {_v}")
else:
    print("No overrides set; using configs/default.yaml as-is.")
pipeline = ExperimentPipeline(config=config, config_path=CONFIG_PATH)
data_module = pipeline.setup_data()

encoder_path = Path(pipeline.checkpoint_mgr.directory) / "encoder_pretrained.pt"
finetune_ckpt = Path(pipeline.checkpoint_mgr.directory) / "finetune_best.pt"

results = {
    "history": {},
    "cifar10c_results": None,
    "artifacts": {},
}
ft_model = None

# --- Stage A ---
if RUN_STAGE_A:
    ssl_train_loader, ssl_val_loader = data_module.ssl_loaders()
    encoder_path = pipeline.run_stage_a(ssl_train_loader, ssl_val_loader)
    results["artifacts"]["encoder_checkpoint"] = str(encoder_path)
else:
    assert encoder_path.exists(), (
        f"Stage A skipped but {encoder_path} is missing. "
        "Enable RESTORE_FROM_DRIVE or set RUN_STAGE_A = True."
    )
    print(f"Stage A skipped; reusing {encoder_path}")

# --- Stage B (1 + 2) ---
if RUN_STAGE_B1 or RUN_STAGE_B2:
    sup_train_loader, sup_val_loader = data_module.supervised_loaders()

if RUN_STAGE_B1:
    lp_history = pipeline.run_stage_b1(encoder_path, sup_train_loader, sup_val_loader)
    results["history"]["linear_probe"] = lp_history
    results["artifacts"]["linear_probe_checkpoint"] = str(
        pipeline.checkpoint_mgr.directory / "linear_probe_best.pt"
    )
else:
    print("Stage B.1 skipped.")

if RUN_STAGE_B2:
    ft_model, ft_history = pipeline.run_stage_b2(
        encoder_path, sup_train_loader, sup_val_loader
    )
    results["history"]["finetune"] = ft_history
    results["artifacts"]["finetune_checkpoint"] = str(finetune_ckpt)
else:
    print("Stage B.2 skipped.")

# --- Stage C ---
if RUN_STAGE_C:
    if ft_model is None:
        assert finetune_ckpt.exists(), (
            f"Stage C requested but {finetune_ckpt} is missing. "
            "Run Stage B.2 or restore the checkpoint from Drive."
        )
        ft_model = pipeline.load_finetuned_model()
    results["cifar10c_results"] = pipeline.run_stage_c(ft_model, data_module)
else:
    print("Stage C skipped.")

pipeline.logger.close()

print("\nArtifacts produced:")
for name, path in results["artifacts"].items():
    print(f"  {name}: {path}")


## 7. Inspect Logs and Metrics


In [ ]:
!ls -R logs 2>/dev/null || echo 'no logs/'
!echo
!ls -R checkpoints 2>/dev/null || echo 'no checkpoints/'


In [ ]:
experiment = config.experiment.name
!sed -n "1,400p" "logs/{experiment}/experiment.log"


In [ ]:
import pandas as pd

metrics_path = Path("logs") / config.experiment.name / "metrics.csv"
if metrics_path.exists():
    df = pd.read_csv(metrics_path, header=None, names=["step", "tag", "value"])
    display(df.tail(60))
else:
    print(f"Metrics file not found: {metrics_path}")


## 8. Summarize CIFAR-10-C Results (only if Stage C ran)


In [ ]:
import pandas as pd

csv_path = Path("logs") / config.experiment.name / "cifar10c_results.csv"
if not csv_path.exists():
    print(f"No CIFAR-10-C report at {csv_path}; Stage C did not run.")
else:
    df = pd.read_csv(csv_path)
    display(df)

    has_ttt = "ttt_accuracy" in df.columns
    df_corr = df[df["corruption"] != "clean"]

    print("\nMean baseline accuracy per severity:")
    display(df_corr.groupby("severity")["baseline_accuracy"].mean())
    if has_ttt:
        print("\nMean TTT accuracy per severity:")
        display(df_corr.groupby("severity")["ttt_accuracy"].mean())
        print("\nMean delta (TTT - baseline) per severity:")
        display(df_corr.groupby("severity")["delta_accuracy"].mean())

    print("\nMean baseline accuracy per corruption:")
    display(df_corr.groupby("corruption")["baseline_accuracy"].mean().sort_values())
    if has_ttt:
        print("\nMean delta per corruption (TTT - baseline):")
        display(df_corr.groupby("corruption")["delta_accuracy"].mean().sort_values(ascending=False))


## 9. TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs


## 10. Persist Artifacts to Google Drive

Mirrors `logs/` and `checkpoints/` to `DRIVE_RUNS_DIR`. Re-running this cell
is safe; existing files are overwritten.


In [ ]:
if USE_GOOGLE_DRIVE:
    runs_dir = Path(DRIVE_RUNS_DIR)
    runs_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree("logs", runs_dir / "logs", dirs_exist_ok=True)
    shutil.copytree("checkpoints", runs_dir / "checkpoints", dirs_exist_ok=True)
    print(f"Saved logs and checkpoints to {runs_dir}")
else:
    print("Set USE_GOOGLE_DRIVE = True at the top of the notebook to enable Drive persistence.")
